# Notebook 03 - Build the Hard Subset `D_hard`

**Argus Vision / ISIC-2019 multi-agent dermoscopy pipeline.**

This notebook loads the two previously-trained agents (Agent A = EfficientNet-B4,
Agent B = ViT-Base) and runs them over the held-out validation/test split. For
every image we compute the **disagreement** between the two agents
(Jensen-Shannon divergence) and each agent's **uncertainty** (Shannon entropy).
Images where the agents disagree strongly *or* are individually uncertain are
collected into the hard subset `D_hard`, which downstream notebooks (NB04 the
consensus/arbiter model, NB05 the evaluation) consume.

---

## Kaggle setup (do this before *Run All*)

1. **Attach the dataset** `andrewmvd/isic-2019` (Add Input -> search *isic-2019* ->
   mounts read-only under `/kaggle/input/isic-2019`).
2. **Attach the NB01 and NB02 outputs** as input datasets. These notebooks save
   `agent_a_best.pth` (NB01) and `agent_b_best.pth` (NB02) to `/kaggle/working/`.
   Save each notebook's output as a dataset (or *Add Input -> Your Work -> Notebook
   Output*) and attach both here so this notebook can find the checkpoints under
   `/kaggle/input/`.
3. **Settings -> Accelerator -> GPU** (T4 x2 or P100).
4. **Settings -> Internet -> ON** (needed for `pip install` and, as a fallback,
   downloading pretrained backbone weights if a checkpoint is missing).
5. **Run All**.

> If a checkpoint cannot be found this notebook will warn you, fall back to
> `pretrained=True` backbones so it still runs end-to-end, and print exactly which
> dataset you need to attach.

**Outputs** are written to `/kaggle/working/`:
`hard_subset.csv` (the fired rows) and `all_scores.csv` (every image's scores),
plus figures. See the final cell for how to download / attach them to NB04 & NB05.


## 1. Install extras, imports and shared helpers

In [ ]:
# --- Install extras (Kaggle pre-installs torch/torchvision/numpy/pandas/sklearn/
# matplotlib/seaborn/opencv/Pillow/tqdm). We add an up-to-date `timm` so the ViT
# model id resolves. scipy is pre-installed on Kaggle.
import sys, subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "timm"],
    check=False,
)
print("NOTE: Kaggle Internet must be ON for pip + (fallback) pretrained weights.")

# --- Imports ---------------------------------------------------------------
import os
import glob
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.spatial.distance import jensenshannon
from sklearn.model_selection import train_test_split

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
import timm

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch={torch.__version__}  timm={timm.__version__}  device={DEVICE}")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ===========================================================================
# SHARED CONTRACT CONSTANTS (kept identical across the whole pipeline)
# ===========================================================================
ISIC_CLASSES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
NUM_CLASSES = 8
IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

BATCH_SIZE = 32   # lower to 16 if you hit OOM on a T4
NUM_WORKERS = 2

# ===========================================================================
# SHARED HELPER: discover the ISIC-2019 ground-truth CSV + image directory
# ===========================================================================
def discover_isic(root="/kaggle/input"):
    """Walk `root` and locate (csv_path, image_dir) for ISIC-2019.

    - csv_path: the .csv whose header contains ALL 8 ISIC class names.
    - image_dir: the directory holding the most ISIC_*.jpg/.jpeg files.
    Robust to nested mirror folders (e.g. doubly-nested Training_Input/).
    """
    class_set = {c.upper() for c in ISIC_CLASSES}
    csv_path = None
    # 1) find the ground-truth CSV by inspecting headers only (nrows=0).
    for dirpath, _dirnames, filenames in os.walk(root):
        for fn in filenames:
            if not fn.lower().endswith(".csv"):
                continue
            fp = os.path.join(dirpath, fn)
            try:
                cols = {c.upper() for c in pd.read_csv(fp, nrows=0).columns}
            except Exception:
                continue
            if class_set.issubset(cols):
                csv_path = fp
                break
        if csv_path is not None:
            break

    # 2) find the image directory with the most ISIC_*.jpg/.jpeg files.
    best_dir, best_count = None, -1
    for dirpath, _dirnames, filenames in os.walk(root):
        count = 0
        for fn in filenames:
            low = fn.lower()
            if low.startswith("isic_") and (low.endswith(".jpg") or low.endswith(".jpeg")):
                count += 1
        if count > best_count:
            best_count, best_dir = count, dirpath
    image_dir = best_dir

    print(f"[discover_isic] csv_path  = {csv_path}")
    print(f"[discover_isic] image_dir = {image_dir}  ({best_count} ISIC_*.jpg/.jpeg)")
    assert csv_path is not None and os.path.isfile(csv_path), \
        "Could not find ISIC GroundTruth CSV. Attach dataset andrewmvd/isic-2019."
    assert image_dir is not None and os.path.isdir(image_dir) and best_count > 0, \
        "Could not find ISIC image directory. Attach dataset andrewmvd/isic-2019."
    return csv_path, image_dir

# ===========================================================================
# SHARED HELPER: find a previously-trained file (checkpoint / csv) by substring
# ===========================================================================
def find_file(filename_substring, search_roots=("/kaggle/input", "/kaggle/working")):
    """Return the first path whose basename contains `filename_substring`.

    Walks the given roots (attached datasets + working dir). Returns None and
    prints a clear message if nothing matches.
    """
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _dirnames, filenames in os.walk(root):
            for fn in filenames:
                if filename_substring in fn:
                    found = os.path.join(dirpath, fn)
                    print(f"[find_file] '{filename_substring}' -> {found}")
                    return found
    print(f"[find_file] '{filename_substring}' NOT FOUND under {search_roots}.")
    return None

print("Helpers ready: discover_isic(), find_file()")


## 2. Discover the dataset and rebuild the held-out split

We reuse the **exact same** stratified split as the training notebooks
(`test_size=0.15`, `stratify=labels`, `random_state=42`) so that the validation
set here matches what Agent A / Agent B were validated against. We run the hard-subset
mining over this held-out split.

In [ ]:
CSV_PATH, IMAGE_DIR = discover_isic()

gt = pd.read_csv(CSV_PATH)
# Map header columns to the canonical class order (case-insensitive).
col_upper = {c.upper(): c for c in gt.columns}
class_cols = [col_upper[c.upper()] for c in ISIC_CLASSES]
assert "image" in {c.lower() for c in gt.columns}, "GroundTruth CSV must have an 'image' column."
image_col = [c for c in gt.columns if c.lower() == "image"][0]

# label = argmax over the 8 canonical one-hot class columns.
labels_all = gt[class_cols].to_numpy().argmax(axis=1).astype(int)
images_all = gt[image_col].astype(str).tolist()
print(f"Total labelled images in CSV: {len(images_all)}")

# Class distribution (full dataset).
full_counts = pd.Series(labels_all).value_counts().reindex(range(NUM_CLASSES), fill_value=0)
print("Full dataset class counts:")
for i, c in enumerate(ISIC_CLASSES):
    print(f"  {c:>4}: {int(full_counts[i])}")

# ---------------------------------------------------------------------------
# torchvision transforms (val/inference pipeline) -- mirrors dataset.py.
# ---------------------------------------------------------------------------
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# (train transforms kept here for reference / parity with NB01-02.)
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])

# ---------------------------------------------------------------------------
# Inline Dataset: returns (tensor, label_int, image_path).
# ---------------------------------------------------------------------------
class ISICDataset(Dataset):
    def __init__(self, image_names, labels, image_dir, transform):
        self.image_names = list(image_names)
        self.labels = list(labels)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def _resolve_path(self, name):
        # CSV stores extension-less names: image path = image_dir/<name>.jpg
        direct = os.path.join(self.image_dir, name)
        if os.path.isfile(direct):
            return direct
        for ext in (".jpg", ".jpeg", ".JPG", ".JPEG", ".png", ".PNG"):
            cand = os.path.join(self.image_dir, name + ext)
            if os.path.isfile(cand):
                return cand
        return os.path.join(self.image_dir, name + ".jpg")

    def __getitem__(self, idx):
        path = self._resolve_path(self.image_names[idx])
        with Image.open(path) as h:
            img = h.convert("RGB")
        tensor = self.transform(img)
        return tensor, int(self.labels[idx]), path

# ---------------------------------------------------------------------------
# Stratified split: identical to the training notebooks.
# ---------------------------------------------------------------------------
train_imgs, val_imgs, train_lbls, val_lbls = train_test_split(
    images_all, labels_all,
    test_size=0.15, stratify=labels_all, random_state=42,
)
print(f"\nSplit -> train={len(train_imgs)}  val/test(held-out)={len(val_imgs)}")

val_dataset = ISICDataset(val_imgs, val_lbls, IMAGE_DIR, val_transforms)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Held-out loader: {len(val_dataset)} images, {len(val_loader)} batches.")


## 3. Load both agents (Agent A = EfficientNet-B4, Agent B = ViT-Base)

We construct each backbone with `num_classes=8` and `pretrained=False`, then load
the checkpoint located via `find_file(...)`. If a checkpoint is missing we warn,
fall back to `pretrained=True` (so the notebook still runs top-to-bottom), and
print exactly which dataset to attach.

In [ ]:
AGENT_A_MODEL = "efficientnet_b4"
AGENT_B_MODEL = "vit_base_patch16_224.augreg_in21k_ft_in1k"

def load_agent(model_name, ckpt_substring):
    """Build `model_name` and load `ckpt_substring`. Fall back to pretrained."""
    ckpt_path = find_file(ckpt_substring)
    if ckpt_path is None:
        print(f"  !! No checkpoint matching '{ckpt_substring}'. Falling back to "
              f"pretrained=True for {model_name}.")
        print(f"  -> To use the real trained agent, attach the notebook output that "
              f"contains '{ckpt_substring}.pth' as an input dataset, then re-run.")
        model = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES)
    else:
        model = timm.create_model(model_name, pretrained=False, num_classes=NUM_CLASSES)
        state = torch.load(ckpt_path, map_location=DEVICE)
        # Unwrap common checkpoint containers.
        if isinstance(state, dict):
            for key in ("state_dict", "model_state_dict", "model"):
                if key in state and isinstance(state[key], dict):
                    state = state[key]
                    break
        # Strip a leading "module." (DataParallel) if present.
        state = { (k[7:] if k.startswith("module.") else k): v for k, v in state.items() }
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f"  loaded '{ckpt_substring}' (missing={len(missing)}, unexpected={len(unexpected)})")
    return model.to(DEVICE).eval()

print("Loading Agent A (EfficientNet-B4):")
agent_a = load_agent(AGENT_A_MODEL, "agent_a_best")
print("\nLoading Agent B (ViT-Base):")
agent_b = load_agent(AGENT_B_MODEL, "agent_b_best")

# Free a little memory and confirm eval mode.
for m, name in ((agent_a, "Agent A"), (agent_b, "Agent B")):
    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"{name}: {n_params:.1f}M params, training={m.training}")


## 4. Run inference: per-image probability vectors for both agents

We run both agents over the held-out split with mixed precision (`autocast`) and
collect softmax probability vectors. The image paths and true labels are kept
aligned so we can write a per-image scores table.

In [ ]:
all_paths = []
all_true = []
probs_a_list = []
probs_b_list = []

amp_enabled = (DEVICE == "cuda")

with torch.no_grad():
    for imgs, labels, paths in tqdm(val_loader, desc="Inference (both agents)"):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits_a = agent_a(imgs)
            logits_b = agent_b(imgs)
        pa = F.softmax(logits_a.float(), dim=1).cpu().numpy()
        pb = F.softmax(logits_b.float(), dim=1).cpu().numpy()
        probs_a_list.append(pa)
        probs_b_list.append(pb)
        all_true.extend([int(x) for x in labels.numpy()])
        all_paths.extend(list(paths))

probs_a = np.concatenate(probs_a_list, axis=0)
probs_b = np.concatenate(probs_b_list, axis=0)
true_labels = np.array(all_true, dtype=int)
pred_a = probs_a.argmax(axis=1)
pred_b = probs_b.argmax(axis=1)

print(f"Collected probabilities: A={probs_a.shape}  B={probs_b.shape}")
print(f"Agent A acc on held-out: {(pred_a == true_labels).mean():.4f}")
print(f"Agent B acc on held-out: {(pred_b == true_labels).mean():.4f}")
print(f"Agents agree on label:   {(pred_a == pred_b).mean():.4f}")


## 5. Disagreement (JS divergence) + uncertainty (Shannon entropy)

For each image we compute:

- **JS divergence** between the two probability vectors:
  `scipy.spatial.distance.jensenshannon(pa, pb, base=2) ** 2`
  (scipy returns the JS *distance* = sqrt of the divergence, so we square it; in
  bits, range `[0, 1]`).
- **Shannon entropy** (base 2) for each agent's distribution.

The hard-subset **trigger fires** for an image when the agents disagree
(`js > 0.25`) **or** either agent is uncertain (`max(entropy_a, entropy_b) > 0.8`).

In [ ]:
JS_THRESHOLD = 0.25
ENTROPY_THRESHOLD = 0.8
EPS = 1e-12

def shannon_entropy_b2(p):
    """Row-wise Shannon entropy in bits for a (N, C) probability matrix."""
    p = np.clip(p, EPS, 1.0)
    return -np.sum(p * np.log2(p), axis=1)

js_div = np.empty(len(true_labels), dtype=np.float64)
for i in range(len(true_labels)):
    # jensenshannon returns the JS *distance* (sqrt of divergence); square it.
    d = jensenshannon(probs_a[i], probs_b[i], base=2)
    js_div[i] = float(d) ** 2 if np.isfinite(d) else 0.0

entropy_a = shannon_entropy_b2(probs_a)
entropy_b = shannon_entropy_b2(probs_b)
max_entropy = np.maximum(entropy_a, entropy_b)

fired = (js_div > JS_THRESHOLD) | (max_entropy > ENTROPY_THRESHOLD)

print(f"JS divergence:  min={js_div.min():.4f}  mean={js_div.mean():.4f}  max={js_div.max():.4f}")
print(f"Entropy A (bits): mean={entropy_a.mean():.4f}  max={entropy_a.max():.4f}")
print(f"Entropy B (bits): mean={entropy_b.mean():.4f}  max={entropy_b.max():.4f}")
print(f"\nTrigger: js>{JS_THRESHOLD} OR max_entropy>{ENTROPY_THRESHOLD}")
print(f"  fired by JS only      : {int(((js_div > JS_THRESHOLD) & ~(max_entropy > ENTROPY_THRESHOLD)).sum())}")
print(f"  fired by entropy only : {int((~(js_div > JS_THRESHOLD) & (max_entropy > ENTROPY_THRESHOLD)).sum())}")
print(f"  fired by both         : {int(((js_div > JS_THRESHOLD) & (max_entropy > ENTROPY_THRESHOLD)).sum())}")
print(f"  TOTAL D_hard size     : {int(fired.sum())} / {len(fired)} "
      f"({100.0 * fired.mean():.1f}% of held-out)")

# Quick distribution plots of the two signals.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(js_div, bins=50, color="#4C72B0", edgecolor="white")
axes[0].axvline(JS_THRESHOLD, color="red", ls="--", label=f"thr={JS_THRESHOLD}")
axes[0].set_title("JS divergence (bits) between agents"); axes[0].set_xlabel("JS"); axes[0].legend()
axes[1].hist(max_entropy, bins=50, color="#55A868", edgecolor="white")
axes[1].axvline(ENTROPY_THRESHOLD, color="red", ls="--", label=f"thr={ENTROPY_THRESHOLD}")
axes[1].set_title("max(entropy_a, entropy_b) (bits)"); axes[1].set_xlabel("entropy"); axes[1].legend()
plt.tight_layout()
os.makedirs("/kaggle/working/figures", exist_ok=True)
plt.savefig("/kaggle/working/figures/hard_subset_signals.png", dpi=120, bbox_inches="tight")
plt.show()


## 6. Save `hard_subset.csv` and `all_scores.csv`

`hard_subset.csv` contains only the **fired** rows (the actual `D_hard`).
`all_scores.csv` contains every held-out image's scores for full reproducibility
and threshold re-tuning later.

In [ ]:
label_names = np.array(ISIC_CLASSES)

scores_df = pd.DataFrame({
    "image_path": all_paths,
    "true_label": label_names[true_labels],
    "pred_a": label_names[pred_a],
    "pred_b": label_names[pred_b],
    "js_divergence": js_div,
    "entropy_a": entropy_a,
    "entropy_b": entropy_b,
    "fired": fired,
})

ALL_SCORES_CSV = "/kaggle/working/all_scores.csv"
HARD_SUBSET_CSV = "/kaggle/working/hard_subset.csv"

scores_df.to_csv(ALL_SCORES_CSV, index=False)

hard_df = (
    scores_df[scores_df["fired"]]
    .drop(columns=["fired"])
    .sort_values("js_divergence", ascending=False)
    .reset_index(drop=True)
)
hard_df.to_csv(HARD_SUBSET_CSV, index=False)

print(f"Saved full scores  -> {ALL_SCORES_CSV}  ({len(scores_df)} rows)")
print(f"Saved D_hard       -> {HARD_SUBSET_CSV}  ({len(hard_df)} rows)")
print("\nColumns:", list(hard_df.columns))
hard_df.head(10)


## 7. Class distribution: `D_hard` vs the full held-out split

Hard cases are not uniformly distributed across classes; this chart shows which
diagnoses the two agents struggle to agree on (often the rarer / visually
ambiguous classes).

In [ ]:
full_dist = pd.Series(label_names[true_labels]).value_counts().reindex(ISIC_CLASSES, fill_value=0)
hard_dist = hard_df["true_label"].value_counts().reindex(ISIC_CLASSES, fill_value=0)

# Proportion of each class that ended up in D_hard.
with np.errstate(divide="ignore", invalid="ignore"):
    hard_rate = (hard_dist / full_dist).fillna(0.0) * 100.0

x = np.arange(NUM_CLASSES)
w = 0.4
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.bar(x - w/2, full_dist.values, width=w, label="Full held-out", color="#4C72B0")
ax1.bar(x + w/2, hard_dist.values, width=w, label="D_hard", color="#C44E52")
ax1.set_xticks(x); ax1.set_xticklabels(ISIC_CLASSES)
ax1.set_ylabel("count"); ax1.set_title("Class counts: full split vs D_hard")
ax1.legend()
for i, v in enumerate(hard_dist.values):
    ax1.text(i + w/2, v, str(int(v)), ha="center", va="bottom", fontsize=8)

ax2.bar(x, hard_rate.values, color="#8172B3")
ax2.set_xticks(x); ax2.set_xticklabels(ISIC_CLASSES)
ax2.set_ylabel("% of class flagged hard"); ax2.set_title("Per-class hard-rate")
for i, v in enumerate(hard_rate.values):
    ax2.text(i, v, f"{v:.0f}%", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("/kaggle/working/figures/hard_subset_class_dist.png", dpi=120, bbox_inches="tight")
plt.show()

print("Per-class D_hard counts:")
for c in ISIC_CLASSES:
    print(f"  {c:>4}: {int(hard_dist[c]):4d} / {int(full_dist[c]):4d}  ({hard_rate[c]:.1f}%)")


## 8. The 20 hardest cases (highest JS divergence)

A visual grid of the images where the two agents disagree most. Each tile shows
the **true** label and both agents' predictions, so you can sanity-check that
`D_hard` is genuinely capturing ambiguous / hard dermoscopy cases.

In [ ]:
n_show = min(20, len(hard_df))
top = hard_df.head(n_show).reset_index(drop=True)

if n_show == 0:
    print("D_hard is empty - no hard cases fired. Try lowering the thresholds.")
else:
    ncols = 5
    nrows = int(math.ceil(n_show / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4.2 * nrows))
    axes = np.array(axes).reshape(-1)
    for i in range(len(axes)):
        ax = axes[i]
        ax.axis("off")
        if i >= n_show:
            continue
        row = top.iloc[i]
        try:
            with Image.open(row["image_path"]) as h:
                img = h.convert("RGB")
            ax.imshow(img)
        except Exception as exc:
            ax.text(0.5, 0.5, f"load error\n{exc}", ha="center", va="center", fontsize=8)
        agree_color = "green" if row["pred_a"] == row["pred_b"] else "red"
        ax.set_title(
            f"true: {row['true_label']}\n"
            f"A={row['pred_a']}  B={row['pred_b']}\n"
            f"JS={row['js_divergence']:.3f}  "
            f"H_a={row['entropy_a']:.2f} H_b={row['entropy_b']:.2f}",
            fontsize=9, color=agree_color,
        )
    plt.suptitle("Top hardest cases by JS divergence (true vs Agent A / Agent B)", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig("/kaggle/working/figures/hard_subset_top20.png", dpi=120, bbox_inches="tight")
    plt.show()


## 9. Outputs and next steps

This notebook wrote everything to `/kaggle/working/`:

| File | Description |
|------|-------------|
| `/kaggle/working/hard_subset.csv` | **`D_hard`** - only the fired rows (`image_path, true_label, pred_a, pred_b, js_divergence, entropy_a, entropy_b`), sorted hardest-first. |
| `/kaggle/working/all_scores.csv` | Every held-out image's scores + a `fired` flag (for re-tuning thresholds). |
| `/kaggle/working/figures/*.png` | Signal histograms, class-distribution and top-20 hard-case grids. |

### Download / hand off to NB04 & NB05

1. After *Run All* finishes, open the **Output** tab (right panel) and download
   `hard_subset.csv` (and `all_scores.csv` if you want them).
2. **Save this notebook's output as a dataset** so the next notebooks can mount it:
   *Save Version* -> the run's output appears under *Your Work*. In **NB04
   (train consensus / arbiter)** and **NB05 (evaluation)**, *Add Input -> Your Work ->*
   select this notebook's output. `hard_subset.csv` will then be discoverable via
   `find_file("hard_subset")` under `/kaggle/input/`.
3. NB04 trains the consensus model on `D_hard`; NB05 evaluates the full
   three-agent system. Both still also need `andrewmvd/isic-2019` and the
   `agent_a_best.pth` / `agent_b_best.pth` checkpoints attached.

> Tip: keep the JS / entropy thresholds (`0.25` / `0.8`) consistent with what NB04
> assumes, or re-derive `D_hard` from `all_scores.csv` if you change them.